# MSCS2202 Machine Learning Kaggle Challenge

## Objective

The goal of this project is to predict the continuous `target` variable using 20 numerical input features (`x0` to `x19`).

This is a supervised regression problem.

The competition is evaluated using the R² (coefficient of determination) metric, where a higher score indicates better predictive performance.

The project explores different preprocessing and regression approaches to determine which model performs best.

In [1]:
# Import libraries for data manipulation
import pandas as pd

# Import train-test split for model validation
from sklearn.model_selection import train_test_split

# Import SimpleImputer for handling missing values
from sklearn.impute import SimpleImputer

# Import regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# Import preprocessing tools
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

# Import Pipeline for combining preprocessing and modeling steps
from sklearn.pipeline import Pipeline

# Import R² score for model evaluation
from sklearn.metrics import r2_score

## Load the Dataset

The competition provides a training dataset containing the target variable and a test dataset where the target must be predicted.

In [2]:
#Load the dataset
train = pd.read_csv("~/Downloads/sofia-ml-regression-2026-summer-private/summer2026_kaggle_linear_regression_challenge_train.csv")
test =pd.read_csv("~/Downloads/sofia-ml-regression-2026-summer-private/summer2026_kaggle_linear_regression_challenge_test.csv")

## Exploratory Data Analysis

The first step was to understand the structure, dimensions, variable types, and distribution of the training and test datasets.

In [3]:
#Display first (5) rows of the train dataset
train.head(5)


,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x12,x13,x14,x15,x16,x17,x18,x19,target,Id
0,6.068698,-0.330868,2.141601,-0.303288,8.382691,-1.911467,-16.510225,6.699953,7.526628,-0.500969,...,15.024857,8.681236,7.176912,3.356824,-10.688157,1.331724,-16.148265,-25.790612,-24.549702,3
1,27.462096,0.633254,8.818176,-2.925587,10.120454,1.936762,-8.206700,9.626216,2.189313,3.671586,...,0.529507,-15.306418,0.294682,-1.847270,13.650131,18.231600,6.593877,-26.834145,132.397451,5
2,6.347459,-4.329401,4.630901,0.293031,-10.921375,-3.540781,11.792638,29.882761,-0.214754,1.194087,...,-16.139854,0.746393,2.611303,-1.103982,1.367624,-4.847642,13.752310,-1.104407,-104.831081,6
3,3.208970,3.765250,2.011389,0.748147,-18.280035,NaN,26.920570,5.506490,4.435851,1.441199,...,-7.786529,5.923576,10.795189,1.136478,3.781332,-9.039253,10.645697,NaN,-154.763244,7
4,25.896142,-1.500400,0.977432,-0.836908,9.531903,-0.464663,-1.745255,-2.161853,-7.012604,3.033782,...,12.997261,1.707714,-37.482851,0.131263,9.434453,5.088502,-6.644931,29.236447,-8.335036,9


In [4]:
#Display rows and columns of the dataset
train.shape

(2500, 22)

In [5]:
#Display column names
train.columns


Index(['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10',
       'x11', 'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x19', 'target',
       'Id'],
      dtype='object')

In [6]:
#check for missing values
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 22 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   x0      2478 non-null   float64
 1   x1      2479 non-null   float64
 2   x2      2475 non-null   float64
 3   x3      2478 non-null   float64
 4   x4      2473 non-null   float64
 5   x5      2469 non-null   float64
 6   x6      2476 non-null   float64
 7   x7      2485 non-null   float64
 8   x8      2480 non-null   float64
 9   x9      2472 non-null   float64
 10  x10     2475 non-null   float64
 11  x11     2474 non-null   float64
 12  x12     2483 non-null   float64
 13  x13     2476 non-null   float64
 14  x14     2479 non-null   float64
 15  x15     2470 non-null   float64
 16  x16     2470 non-null   float64
 17  x17     2471 non-null   float64
 18  x18     2475 non-null   float64
 19  x19     2471 non-null   float64
 20  target  2500 non-null   float64
 21  Id      2500 non-null   int64  
dtype

In [7]:
train.describe()

,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x12,x13,x14,x15,x16,x17,x18,x19,target,Id
count,2478.000000,2479.000000,2475.000000,2478.000000,2473.000000,2469.000000,2476.000000,2485.000000,2480.000000,2472.000000,...,2483.000000,2476.000000,2479.000000,2470.000000,2470.000000,2471.000000,2475.000000,2471.000000,2500.000000,2500.000000
mean,0.370076,0.073406,0.171521,-0.011232,-0.119634,-0.040149,0.100882,0.202261,-0.034692,-0.066104,...,-0.073400,0.126430,0.007721,-0.038017,0.185633,0.159677,0.293859,-0.399536,1.777084,2510.861600
std,12.180983,3.923268,10.458479,1.868386,9.757498,2.688212,14.163533,14.376643,4.289606,3.651133,...,9.698482,9.189245,13.778775,3.546826,12.834292,13.306741,10.346789,14.394250,149.000936,1438.515953
min,-41.101992,-11.863109,-44.495944,-6.713979,-34.224046,-10.245958,-47.954815,-55.423112,-15.795069,-14.736286,...,-35.565901,-29.498305,-47.448297,-11.579008,-50.306380,-40.541257,-37.318420,-49.885455,-1215.472584,3.000000
25%,-7.708991,-2.608902,-7.006155,-1.300097,-6.919555,-1.851412,-9.780362,-9.637381,-2.860445,-2.458617,...,-6.756101,-6.180529,-9.297222,-2.526713,-8.252778,-8.861684,-6.647366,-9.707911,-75.011431,1272.750000
50%,0.629896,0.068784,0.375571,-0.021830,0.224000,-0.114095,-0.166829,0.267809,-0.058034,-0.073683,...,-0.094734,-0.070685,-0.141617,-0.100835,0.167756,0.389680,0.346459,-0.648510,-0.101003,2544.500000
75%,8.377939,2.681841,7.259837,1.284668,6.622266,1.746053,9.465421,9.620466,2.912471,2.269034,...,6.704773,6.564543,9.481197,2.421301,8.923480,9.195233,7.355855,9.573498,72.746520,3760.000000
max,45.473105,13.916111,33.730027,6.205738,27.217847,8.186229,59.323011,52.876684,15.874072,12.168756,...,31.076026,27.818258,52.551748,12.248139,39.194808,42.598362,40.345193,49.209295,968.431174,4999.000000


In [8]:
#Display first (5) rows of the test dataset
test.head(5)

,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x11,x12,x13,x14,x15,x16,x17,x18,x19,Id
0,-19.541338,0.251163,7.835783,0.282576,8.649919,7.815808,-20.794369,13.328533,-7.217923,1.260250,...,7.401634,-8.300306,4.681126,-17.370138,-7.733925,5.551981,23.537351,5.347965,-14.521484,0
1,3.269554,3.006026,12.599081,-2.142957,6.972854,0.942758,-0.455802,0.185824,-2.942602,-2.275020,...,1.447264,-4.645537,-22.453385,-12.035023,-1.810708,-16.386825,-18.065309,8.492769,-3.582171,1
2,-20.709421,-5.231539,-3.169028,2.064075,-15.085853,4.266250,-6.852488,-24.121770,2.222776,5.268725,...,3.627777,-3.067134,-0.098924,22.865793,3.208661,-15.358403,38.017777,-10.499832,12.289157,2
3,11.195211,-0.584093,10.635770,0.242581,-7.741640,7.765390,19.363690,2.417049,0.096356,6.059145,...,8.524414,6.315878,-11.910904,10.204591,4.002400,6.978826,13.091441,14.135330,-7.118739,4
4,6.074311,-1.872913,21.688982,-0.833080,1.669815,-2.958456,-11.417641,-0.644289,-3.563414,0.461862,...,0.681104,0.269671,11.099658,-6.931203,-3.940650,-10.243780,-29.383008,-22.502817,14.578354,8


In [9]:
#Display rows and columns of the dataset
test.shape

(2500, 21)

In [10]:
#Display column names
test.columns

Index(['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10',
       'x11', 'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x19', 'Id'],
      dtype='object')

In [11]:
#check for missing values
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 21 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   x0      2472 non-null   float64
 1   x1      2473 non-null   float64
 2   x2      2486 non-null   float64
 3   x3      2472 non-null   float64
 4   x4      2476 non-null   float64
 5   x5      2474 non-null   float64
 6   x6      2473 non-null   float64
 7   x7      2476 non-null   float64
 8   x8      2480 non-null   float64
 9   x9      2468 non-null   float64
 10  x10     2477 non-null   float64
 11  x11     2476 non-null   float64
 12  x12     2475 non-null   float64
 13  x13     2473 non-null   float64
 14  x14     2477 non-null   float64
 15  x15     2471 non-null   float64
 16  x16     2473 non-null   float64
 17  x17     2478 non-null   float64
 18  x18     2475 non-null   float64
 19  x19     2466 non-null   float64
 20  Id      2500 non-null   int64  
dtypes: float64(20), int64(1)
memory usage

#check the number of missing values in each column
train.isnull().sum()

In [12]:
#seperate predictors from target
x = train.drop(columns= ["target", "Id"])

#store the target variable
y = train["target"]

# Display the first five rows of the predictor variables
x.head()

,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14,x15,x16,x17,x18,x19
0,6.068698,-0.330868,2.141601,-0.303288,8.382691,-1.911467,-16.510225,6.699953,7.526628,-0.500969,13.442490,-0.493262,15.024857,8.681236,7.176912,3.356824,-10.688157,1.331724,-16.148265,-25.790612
1,27.462096,0.633254,8.818176,-2.925587,10.120454,1.936762,-8.206700,9.626216,2.189313,3.671586,5.649027,-3.099661,0.529507,-15.306418,0.294682,-1.847270,13.650131,18.231600,6.593877,-26.834145
2,6.347459,-4.329401,4.630901,0.293031,-10.921375,-3.540781,11.792638,29.882761,-0.214754,1.194087,-4.327341,-1.511456,-16.139854,0.746393,2.611303,-1.103982,1.367624,-4.847642,13.752310,-1.104407
3,3.208970,3.765250,2.011389,0.748147,-18.280035,NaN,26.920570,5.506490,4.435851,1.441199,NaN,-4.183797,-7.786529,5.923576,10.795189,1.136478,3.781332,-9.039253,10.645697,NaN
4,25.896142,-1.500400,0.977432,-0.836908,9.531903,-0.464663,-1.745255,-2.161853,-7.012604,3.033782,-9.646549,12.368219,12.997261,1.707714,-37.482851,0.131263,9.434453,5.088502,-6.644931,29.236447


In [13]:
# Display the first five target values
y.head()

0    -24.549702
1    132.397451
2   -104.831081
3   -154.763244
4     -8.335036
Name: target, dtype: float64

In [14]:
# Check the dimensions of the predictors and target

print(x.shape)
print(y.shape)

(2500, 20)
(2500,)


## Train and Validation Split

The training dataset was divided into training and validation sets.

80% of the observations were used to train the models and 20% were reserved for validation.

The split was performed before imputation to reduce the risk of data leakage.

In [15]:
# Split the dataset into 80% training data and 20% validation data

x_train, x_valid, y_train, y_valid = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

# Check the shapes of the training and validation datasets
print(x_train.shape)
print(x_valid.shape)
print(y_train.shape)
print(y_valid.shape)


(2000, 20)
(500, 20)
(2000,)
(500,)


## Experiment 1: Linear Regression with Median Imputation

The first experiment used median imputation to replace missing feature values before training a Linear Regression model. Median imputation was selected as the initial baseline because it provides a simple way to handle missing numerical values and is less sensitive to extreme values.

In [16]:
# Create an imputer that will replace missing values with the median of each column
imputer = SimpleImputer(strategy="median")

# Fit the imputer on the training data and transform the training data
x_train_imputed = imputer.fit_transform(x_train)

# Use the same median values learned from the training data to transform the validation data
x_valid_imputed = imputer.transform(x_valid)

In [17]:
# Check the number of missing values after imputation
print("Missing values in training data:", sum(sum(pd.isnull(x_train_imputed))))
print("Missing values in validation data:", sum(sum(pd.isnull(x_valid_imputed))))

Missing values in training data: 0
Missing values in validation data: 0


In [18]:
# Import the Linear Regression model
from sklearn.linear_model import LinearRegression

# Create the Linear Regression model
model = LinearRegression()

# Train the model using the imputed training data
model.fit(x_train_imputed, y_train)

LinearRegression()

In [19]:
# Use the trained model to predict the target values for the validation data
y_pred = model.predict(x_valid_imputed)

# Display the first 10 predicted values
print(y_pred[:10])

[ -22.01646408 -105.29564906 -112.05433628 -122.78665403  -94.23433231
  -51.6047165   110.87046627 -112.08589163 -176.91722221    4.72991254]


In [20]:

# Calculate the R² score by comparing actual values with predicted values
r2 = r2_score(y_valid, y_pred)

# Display the R² score
print("R² Score:", r2)

R² Score: 0.5073640674346939


In [21]:
#Preparing the test features
# Remove the Id column from the test data to create the test features
x_test = test.drop(columns=["Id"])

# Check the shape of the test features
print(x_test.shape)

(2500, 20)


In [22]:
# Create a new median imputer for the final model
final_imputer = SimpleImputer(strategy="median")

# Fit the imputer on all training features and fill the missing values
x_imputed = final_imputer.fit_transform(x)

# Use the same median values to fill missing values in the Kaggle test data
x_test_imputed = final_imputer.transform(x_test)

In [23]:
# Create the final Linear Regression model
final_model = LinearRegression()

# Train the model using all available training data
final_model.fit(x_imputed, y)

LinearRegression()

In [24]:
# Predict the target values for the Kaggle test dataset
test_predictions = final_model.predict(x_test_imputed)

# Display the first 10 predictions
print(test_predictions[:10])

[ 246.97541784  -17.24299507   87.85473061   31.43465739 -333.7206034
  129.96096423   33.86169744  111.24723357 -218.80358559  -99.25814269]


In [25]:
# Create the Kaggle submission file using the test Id and predicted target values
submission = pd.DataFrame({
    "Id": test["Id"],
    "target": test_predictions
})

# Display the first five rows of the submission
submission.head()

,Id,target
0,0,246.975418
1,1,-17.242995
2,2,87.854731
3,4,31.434657
4,8,-333.720603


In [26]:
# Save the predictions as a CSV file for Kaggle submission
submission.to_csv("submission.csv", index=False)

# Confirm that the submission file was created
print("Submission file created successfully.")

Submission file created successfully.


In [27]:
# Display the first five rows of the final submission
submission.head()

,Id,target
0,0,246.975418
1,1,-17.242995
2,2,87.854731
3,4,31.434657
4,8,-333.720603


In [28]:
# Check that the submission has 2,500 rows and 2 columns
print(submission.shape)

# Check that there are no missing values in the submission
print(submission.isnull().sum())

(2500, 2)
Id        0
target    0
dtype: int64


## Experiment 2: Linear Regression with Mean Imputation

Mean imputation was tested to determine whether using the average value of each feature would improve performance compared with median imputation.

In [29]:
# Create a new imputer that replaces missing values with the mean of each column
mean_imputer = SimpleImputer(strategy="mean")

# Fit the mean imputer on the training split and transform the training data
x_train_mean = mean_imputer.fit_transform(x_train)

# Transform the validation data using the mean values learned from the training split
x_valid_mean = mean_imputer.transform(x_valid)

# Create a new Linear Regression model for the mean-imputation experiment
mean_model = LinearRegression()

# Train the model using the mean-imputed training data
mean_model.fit(x_train_mean, y_train)

# Predict the target values for the validation data
y_pred_mean = mean_model.predict(x_valid_mean)

# Calculate the R² score for the mean-imputation model
mean_r2 = r2_score(y_valid, y_pred_mean)

# Display the R² score
print("Mean Imputation R² Score:", mean_r2)

Mean Imputation R² Score: 0.5073844125590137


## Experiment 3: Ridge Regression

Ridge Regression was tested to determine whether L2 regularization could improve the performance of the Linear Regression model by reducing the effect of large coefficients.

The first Ridge model used an alpha value of 1.0.

In [30]:
# Import Ridge Regression
from sklearn.linear_model import Ridge

# Create a Ridge Regression model
ridge_model = Ridge(alpha=1.0)

# Train the Ridge model using the mean-imputed training data
ridge_model.fit(x_train_mean, y_train)

# Predict the validation target values
ridge_pred = ridge_model.predict(x_valid_mean)

# Calculate the R² score for Ridge Regression
ridge_r2 = r2_score(y_valid, ridge_pred)

# Display the Ridge Regression R² score
print("Ridge Regression R² Score:", ridge_r2)

Ridge Regression R² Score: 0.5073838091436386


### Ridge Regression with Feature Scaling

Because Ridge Regression penalizes the size of model coefficients, StandardScaler was applied to place the features on a comparable scale before fitting the model.

In [31]:
# Import StandardScaler to put all features on a similar scale
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and the Ridge model
from sklearn.pipeline import Pipeline

# Create a pipeline that scales the features and then applies Ridge Regression
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

# Train the Ridge pipeline using the mean-imputed training data
ridge_pipeline.fit(x_train_mean, y_train)

# Predict the validation target values
ridge_scaled_pred = ridge_pipeline.predict(x_valid_mean)

# Calculate the R² score for scaled Ridge Regression
ridge_scaled_r2 = r2_score(y_valid, ridge_scaled_pred)

# Display the R² score
print("Scaled Ridge Regression R² Score:", ridge_scaled_r2)

Scaled Ridge Regression R² Score: 0.5073265529695118


### Ridge Hyperparameter Tuning

Different values of alpha were evaluated to determine the regularization strength that produced the highest validation R² score.

In [32]:
# Create a list of different Ridge alpha values to test
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]

# Create an empty list to store each alpha and its R² score
ridge_results = []

# Test each alpha value one at a time
for alpha in alpha_values:

    # Create a pipeline that scales the data and applies Ridge Regression
    ridge_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    # Train the model using the mean-imputed training data
    ridge_pipeline.fit(x_train_mean, y_train)

    # Predict the validation target values
    ridge_pred = ridge_pipeline.predict(x_valid_mean)

    # Calculate the R² score for this alpha value
    ridge_score = r2_score(y_valid, ridge_pred)

    # Store the alpha value and its corresponding R² score
    ridge_results.append((alpha, ridge_score))

    # Display the result for this alpha value
    print(f"Alpha: {alpha} | R² Score: {ridge_score}")

Alpha: 0.001 | R² Score: 0.5073843547718122
Alpha: 0.01 | R² Score: 0.5073838346804677
Alpha: 0.1 | R² Score: 0.5073786331206878
Alpha: 1 | R² Score: 0.5073265529695118
Alpha: 10 | R² Score: 0.5067993755273372
Alpha: 100 | R² Score: 0.5009632957097896


In [33]:
# Find the alpha value with the highest R² score
best_ridge = max(ridge_results, key=lambda x: x[1])

# Display the best alpha and its R² score
print("Best Ridge Alpha:", best_ridge[0])
print("Best Ridge R² Score:", best_ridge[1])

Best Ridge Alpha: 0.001
Best Ridge R² Score: 0.5073843547718122


## Experiment 4: Lasso Regression

Lasso Regression was tested because L1 regularization can shrink some feature coefficients toward zero. This was explored to determine whether reducing the influence of less useful features could improve model performance.

Different alpha values were tested and compared using the validation R² score.

In [34]:
# Import Lasso Regression
from sklearn.linear_model import Lasso

# Create a list of different alpha values to test
lasso_alpha_values = [0.001, 0.01, 0.1, 1, 10]

# Create an empty list to store the results
lasso_results = []

# Test each alpha value
for alpha in lasso_alpha_values:

    # Create a pipeline that scales the features and applies Lasso Regression
    lasso_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=alpha, max_iter=10000))
    ])

    # Train the Lasso model using the mean-imputed training data
    lasso_pipeline.fit(x_train_mean, y_train)

    # Predict the validation target values
    lasso_pred = lasso_pipeline.predict(x_valid_mean)

    # Calculate the R² score
    lasso_score = r2_score(y_valid, lasso_pred)

    # Store the alpha value and its R² score
    lasso_results.append((alpha, lasso_score))

    # Display the result for each alpha value
    print(f"Alpha: {alpha} | R² Score: {lasso_score}")

Alpha: 0.001 | R² Score: 0.5073809931600787
Alpha: 0.01 | R² Score: 0.5073501531312627
Alpha: 0.1 | R² Score: 0.5070356005334491
Alpha: 1 | R² Score: 0.5032007115791537
Alpha: 10 | R² Score: 0.42981239788231795


In [35]:
# Find the Lasso alpha value that produced the highest R² score
best_lasso = max(lasso_results, key=lambda x: x[1])

# Display the best Lasso alpha and R² score
print("Best Lasso Alpha:", best_lasso[0])
print("Best Lasso R² Score:", best_lasso[1])

Best Lasso Alpha: 0.001
Best Lasso R² Score: 0.5073809931600787


## Experiment 5: Elastic Net Regression

Elastic Net combines L1 regularization from Lasso and L2 regularization from Ridge.

Different alpha and L1 ratio combinations were tested to determine whether combining both regularization techniques could improve validation performance.

In [36]:
# Import Elastic Net Regression
from sklearn.linear_model import ElasticNet

# Create different alpha values to test
elastic_alpha_values = [0.001, 0.01, 0.1, 1]

# Create different l1_ratio values to control the mix of Lasso and Ridge
elastic_l1_ratios = [0.2, 0.5, 0.8]

# Create an empty list to store the Elastic Net results
elastic_results = []

# Test each combination of alpha and l1_ratio
for alpha in elastic_alpha_values:
    for l1_ratio in elastic_l1_ratios:

        # Create a pipeline that scales the features and applies Elastic Net
        elastic_pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("elasticnet", ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=10000
            ))
        ])

        # Train the Elastic Net model
        elastic_pipeline.fit(x_train_mean, y_train)

        # Predict the validation target values
        elastic_pred = elastic_pipeline.predict(x_valid_mean)

        # Calculate the R² score
        elastic_score = r2_score(y_valid, elastic_pred)

        # Store the alpha, l1_ratio, and R² score
        elastic_results.append((alpha, l1_ratio, elastic_score))

        # Display the result for each combination
        print(
            f"Alpha: {alpha} | "
            f"L1 Ratio: {l1_ratio} | "
            f"R² Score: {elastic_score}"
        )

Alpha: 0.001 | L1 Ratio: 0.2 | R² Score: 0.5072910810897184
Alpha: 0.001 | L1 Ratio: 0.5 | R² Score: 0.5073248386302392
Alpha: 0.001 | L1 Ratio: 0.8 | R² Score: 0.5073585475781124
Alpha: 0.01 | L1 Ratio: 0.2 | R² Score: 0.5064344554553746
Alpha: 0.01 | L1 Ratio: 0.5 | R² Score: 0.5067817712659712
Alpha: 0.01 | L1 Ratio: 0.8 | R² Score: 0.5071243958163736
Alpha: 0.1 | L1 Ratio: 0.2 | R² Score: 0.49651091966632055
Alpha: 0.1 | L1 Ratio: 0.5 | R² Score: 0.5007462288228604
Alpha: 0.1 | L1 Ratio: 0.8 | R² Score: 0.5046542278756413
Alpha: 1 | L1 Ratio: 0.2 | R² Score: 0.37586893114547315
Alpha: 1 | L1 Ratio: 0.5 | R² Score: 0.4206618343827274
Alpha: 1 | L1 Ratio: 0.8 | R² Score: 0.4717588555613548


In [37]:
# Find the Elastic Net combination with the highest R² score
best_elastic = max(elastic_results, key=lambda x: x[2])

# Display the best Elastic Net settings and R² score
print("Best Elastic Net Alpha:", best_elastic[0])
print("Best Elastic Net L1 Ratio:", best_elastic[1])
print("Best Elastic Net R² Score:", best_elastic[2])


Best Elastic Net Alpha: 0.001
Best Elastic Net L1 Ratio: 0.8
Best Elastic Net R² Score: 0.5073585475781124


## Experiment 6: Polynomial Features with Ridge Regression

Polynomial features were introduced to test whether nonlinear effects or interactions between features could improve prediction.

Degree-2 polynomial features were combined with Ridge Regression to control the increased model complexity.

In [38]:
# Import PolynomialFeatures to create squared and interaction features
from sklearn.preprocessing import PolynomialFeatures

# Create different Ridge alpha values to test with polynomial features
poly_alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]

# Create an empty list to store the results
poly_results = []

# Test each Ridge alpha value
for alpha in poly_alpha_values:

    # Create a pipeline that:
    # 1. Adds degree-2 polynomial and interaction features
    # 2. Scales the new features
    # 3. Applies Ridge Regression
    poly_model = Pipeline([
        ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    # Train the polynomial regression model
    poly_model.fit(x_train_mean, y_train)

    # Predict the target values for the validation dataset
    poly_pred = poly_model.predict(x_valid_mean)

    # Calculate the R² score
    poly_score = r2_score(y_valid, poly_pred)

    # Store the alpha value and corresponding R² score
    poly_results.append((alpha, poly_score))

    # Display the result for each alpha value
    print(f"Alpha: {alpha} | Polynomial R² Score: {poly_score}")

Alpha: 0.001 | Polynomial R² Score: 0.4664589360175385
Alpha: 0.01 | Polynomial R² Score: 0.46645908745931175
Alpha: 0.1 | Polynomial R² Score: 0.4664606000203608
Alpha: 1 | Polynomial R² Score: 0.46647554023014814
Alpha: 10 | Polynomial R² Score: 0.4666066667316233
Alpha: 100 | Polynomial R² Score: 0.46632635060992655


In [39]:
# Find the polynomial model with the highest R² score
best_poly = max(poly_results, key=lambda x: x[1])

# Display the best alpha value and R² score
print("Best Polynomial Ridge Alpha:", best_poly[0])
print("Best Polynomial R² Score:", best_poly[1])

Best Polynomial Ridge Alpha: 10
Best Polynomial R² Score: 0.4666066667316233


In [40]:

# Create a mean imputer for the final Kaggle model
final_mean_imputer = SimpleImputer(strategy="mean")

# Fit the imputer on all training features and fill missing values
x_mean_imputed = final_mean_imputer.fit_transform(x)

# Fill missing values in the Kaggle test data using the training means
x_test_mean_imputed = final_mean_imputer.transform(x_test)

# Create a new Linear Regression model
final_mean_model = LinearRegression()

# Train the model using all of the mean-imputed training data
final_mean_model.fit(x_mean_imputed, y)

# Predict the target values for the Kaggle test dataset
mean_test_predictions = final_mean_model.predict(x_test_mean_imputed)

# Create the second Kaggle submission
submission_mean = pd.DataFrame({
    "Id": test["Id"],
    "target": mean_test_predictions
})

# Preview the first five rows
submission_mean.head()

,Id,target
0,0,247.030514
1,1,-17.173383
2,2,87.909200
3,4,31.505998
4,8,-333.712499


## Model Comparison

The regression approaches were compared using the same training-validation split and the R² evaluation metric.

Linear Regression with Mean Imputation produced the highest validation score. Ridge, Lasso, and Elastic Net produced very similar results but did not improve the model. Polynomial features reduced validation performance.

In [41]:
# Create a table comparing the best validation R² score from each experiment
model_results = pd.DataFrame({
    "Model": [
        "Linear Regression + Median Imputation",
        "Linear Regression + Mean Imputation",
        "Ridge Regression",
        "Lasso Regression",
        "Elastic Net",
        "Polynomial Ridge"
    ],
    "R² Score": [
        r2,
        mean_r2,
        best_ridge[1],
        best_lasso[1],
        best_elastic[2],
        best_poly[1]
    ]
})

# Sort the models from highest to lowest R² score
model_results = model_results.sort_values(
    by="R² Score",
    ascending=False
)

# Display the model comparison table
model_results

,Model,R² Score
1,Linear Regression + Mean Imputation,0.507384
2,Ridge Regression,0.507384
3,Lasso Regression,0.507381
0,Linear Regression + Median Imputation,0.507364
4,Elastic Net,0.507359
5,Polynomial Ridge,0.466607
